# Business Entity Resolution · Amazon ML Challenge 2026

**Notebook owner:** not set  
**Pipeline:** country partitions → multiple retrieval channels → hard-negative classifier → calibration and match-set decisions → audited TSV output.

Start with **demo mode**, which runs offline on generated records. Switch to `full` only after the smoke test passes. Full mode indexes **all provided S2/S3 records**, trains on a bounded country-stratified anchor sample, and measures retrieval against the complete target pool. No external business data, APIs, geocoding, or pretrained models are used. Full mode defaults to training-only inspection.

The code supports SageMaker Studio JupyterLab or Jupyter on EC2. It does not provision AWS resources. Keep data, caches, model artifacts, and executed outputs off the public GitHub repository.

## 1. Install in the notebook environment

Use Python 3.11–3.13. Run the following command in a terminal or a separate cell **once**, then restart the kernel:

```python
%pip install -r ../requirements.txt
```

If your notebook starts in the repository root, use `requirements.txt` instead. Do not reinstall packages halfway through a run. The notebook checks the installed versions below.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, time, importlib.metadata

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ber").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "ber").is_dir(), "Start Jupyter in the repository or its notebooks directory"
sys.path.insert(0, str(PROJECT_ROOT))
# Set thread limits before importing numerical libraries. Tune for the AWS instance.
THREADS = int(os.environ.get("BER_THREADS", "2"))
os.environ.setdefault("OMP_NUM_THREADS", str(THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(THREADS))
for package in ["numpy", "scikit-learn", "rapidfuzz", "nbformat"]:
    print(package, importlib.metadata.version(package))
print("Python:", sys.version.split()[0], "CPU cores:", os.cpu_count())

## 2. Your configuration

`MODE="demo"` is the tested safe starting point. In full mode, point `DATA_ROOT` at the folder containing `train/`. `TRAIN_ONLY=True` prevents official test access; only switch it off when ready for inference. The large profile targets 200,000 sampled anchors per country and requires a measured memory/disk budget. A persistent EBS-backed directory is preferable for work artifacts. Set an existing S3 object URI only if you want the optional input-download cell to run. Credentials come from your IAM role; never paste access keys into this notebook.

Keep a new `RUN_NAME` for each retrieval/feature/training experiment. The cache refuses incompatible reuse. Candidate caps trade recall for runtime; evaluate them before increasing model complexity.

In [ ]:
MODE = os.environ.get("BER_MODE", "demo")
TRAIN_ONLY = MODE != "demo"  # official test data stays unopened until deliberately disabled
PROFILE = "pilot"           # "pilot" or "large"; large is not yet benchmarked on AWS
TEAM_NAME = ""
TEAM_MEMBERS = ""
RUN_NAME = "robust_v3_" + PROFILE
WORK_ROOT = Path(os.environ.get("BER_WORK_ROOT", str(PROJECT_ROOT / "work"))) / MODE / "v3"
DATA_ROOT = Path(os.environ.get("BER_DATA_ROOT", str(WORK_ROOT / "data/student_resource/dataset")))
S3_INPUT_URI = ""
S3_OUTPUT_URI = ""
PER_COUNTRY = 160 if MODE == "demo" else (200_000 if PROFILE == "large" else 1000)
MAX_TRAINING_PAIRS = 100_000 if MODE == "demo" else (140_000_000 if PROFILE == "large" else 700_000)
MAX_ITER = 35 if MODE == "demo" else 180
USE_MINED_LANGUAGE = True
RUN_GEOGRAPHIC_STRESS = True
EVALUATE_TRUST = MODE == "demo"   # enable once after selecting settings on calibration
RUN_FULL_TEST = MODE == "demo"
RUN_FALLBACK = False
BENCHMARK_ANCHORS = 3000
WORKERS = 1 if MODE == "demo" else min(4, max(1, (os.cpu_count() or 2) // 2))
INFERENCE_BUDGET_HOURS = 12.
ALLOW_RUNTIME_OVERRUN = False
PACKAGE_SUBMISSION = False
UPLOAD_PACKAGE_TO_S3 = False
RUN_OFFICIAL_VALIDATOR = MODE == "full" and not TRAIN_ONLY
OFFICIAL_VALIDATOR = DATA_ROOT.parent / "utils" / "validate_submission.py"

assert MODE in {"demo", "full"} and PROFILE in {"pilot", "large"}
assert not (TRAIN_ONLY and RUN_FULL_TEST), "Disable TRAIN_ONLY explicitly before official inference"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = WORK_ROOT / "runs" / RUN_NAME
TRAIN_INDEX = RUN_DIR / "index_train"
TEST_INDEX = RUN_DIR / "index_test"
OUTPUT_DIR = RUN_DIR / "output"
LANGUAGE_PATH = RUN_DIR / "language.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Work:", WORK_ROOT, "Free GiB:", round(shutil.disk_usage(WORK_ROOT).free / 2**30, 1))
print("Large profile: ~120k training + ~20k tuning anchors per country, before grouped rounding.")
print("Estimated feature capacity GiB:", round(MAX_TRAINING_PAIRS * 4 * 100 / 2**30, 1))


## 3. Load the provided resource, or create the demo

Full-data indexes may be much larger than the 1 GB ZIP. Leave substantial disk headroom for FTS indexes, pairs, shards and candidate exports; inspect growth during index construction. Test one country or a small fixture before committing a machine. This implementation has not been benchmarked on the full AWS workload.

In [ ]:
from ber.aws import download, extract_resource
from ber.demo import make_demo
from ber.common import records, write_json, audit_inputs, file_hash
from ber.retrieval import build_index, RetrievalConfig
from ber.training import prepare_pairs, train, evaluate_trust
from ber.inference import predict, validate, package_submission, fallback
from ber.language import mine_language
from ber.experiments import official_validate, register_run, license_inventory

if MODE == "demo":
    DATA_ROOT = make_demo(WORK_ROOT / "demo_data", train_per_country=PER_COUNTRY)
elif S3_INPUT_URI and (not DATA_ROOT.exists() or (not TRAIN_ONLY and not (DATA_ROOT / "test/test_source1.tsv").exists())):
    archive = WORK_ROOT / "student_resource.zip"
    download(S3_INPUT_URI, archive)
    DATA_ROOT = extract_resource(archive, WORK_ROOT / "data", train_only=TRAIN_ONLY)
input_audit = audit_inputs(DATA_ROOT, full=MODE == "full", train_only=TRAIN_ONLY)
write_json(RUN_DIR / "input_audit.json", input_audit)
input_audit


## 4. Index the complete training target pool

Exact normalized name, legal-suffix-reduced name and exact address channels are combined with BM25 name/address and character-trigram retrieval. Unicode combining marks are preserved. The index is partitioned by observed country labels, including countries never seen during training.

If interrupted during index construction, rerun this cell; completed whole-index manifests are reusable, but an unfinished index is rebuilt. Prediction batches later support fine-grained resume.

In [ ]:
language_file = None
if USE_MINED_LANGUAGE:
    if not LANGUAGE_PATH.exists():
        mine_language(DATA_ROOT / "train/train_source1.tsv", DATA_ROOT / "train/train_ground_truth.tsv",
                      [DATA_ROOT / "train" / f"train_source{s}.tsv" for s in (2, 3)],
                      LANGUAGE_PATH, per_country=PER_COUNTRY)
    language_file = LANGUAGE_PATH
training_index_stats = build_index(
    [DATA_ROOT / "train" / f"train_source{s}.tsv" for s in (2, 3)], TRAIN_INDEX, language_file)
training_index_stats


## 5. Create hard negatives and frozen entity splits

The deterministic 60/20/20 split groups country + normalized name + address and stratifies by country and match-count bins. Training anchors and all their labeled links stay in one partition; the entire target pool remains searchable as distractors. This is an exact-signature grouping baseline, **not full near-duplicate clustering**.

There are separate training, calibration and trust partitions. Calibration is further divided into probability calibration and threshold/gate selection. No hidden true links are injected into the candidate set: missed blocking links remain genuine misses in evaluation.

In [ ]:
retrieval = (RetrievalConfig(per_channel=8, per_source=16, max_per_source=24)
             if MODE == "demo" else RetrievalConfig(per_channel=30, per_source=80, max_per_source=160))
pair_stats = prepare_pairs(DATA_ROOT / "train/train_source1.tsv",
                           DATA_ROOT / "train/train_ground_truth.tsv",
                           TRAIN_INDEX, RUN_DIR, retrieval=retrieval,
                           per_country=PER_COUNTRY, seed=2026, max_pairs=MAX_TRAINING_PAIRS)
pair_stats

## 6. Train, calibrate and choose the match-set policy

The classifier uses string similarity, address and numeric evidence, missingness, source indicators, retrieval channels and candidate context. Candidate weights keep large blocks from overwhelming small anchors. The empty-set gate and pair threshold are selected by country-weighted macro F0.5 on the tuning slice.

The default policy uses conditioned thresholds and a learned anchor-level gate; unseen countries use a maximin policy selected on US/India calibration results. An independent-Bernoulli utility decoder is available as an optional experiment. It does not assume one match per source and does not enforce exclusive target ownership. A serialized model should be loaded only from your own trusted runs.

In [ ]:
from ber.training import load_model
MODEL_PATH = RUN_DIR / "model.pkl"
if MODEL_PATH.exists():
    saved = load_model(MODEL_PATH)
    if (saved["pairs_signature"] != pair_stats["signature"]
            or saved["model"].max_iter != MAX_ITER or saved["seed"] != 2026):
        raise ValueError("Saved training settings differ. Choose a new RUN_NAME.")
    tuning = json.loads(MODEL_PATH.with_suffix(".metrics.json").read_text())["tuning"]
else:
    MODEL_PATH, tuning = train(RUN_DIR, max_iter=MAX_ITER, seed=2026)
print("Saved model:", MODEL_PATH)
{key: value for key, value in tuning.items() if key != "trials"}

## 7. Evaluate the untouched trust split once per selected experiment

Read macro F0.5 together with per-country scores, blocking recall, complete-set retrieval, the oracle blocking ceiling, singleton false merges and candidate-count tails. France has no supplied training labels; **none of these numbers estimates French accuracy**. Repeated selection using this trust result turns it into another tuning set.

In [ ]:
trust_metrics = evaluate_trust(RUN_DIR, MODEL_PATH) if EVALUATE_TRUST else {"status": "not opened"}
write_json(RUN_DIR / "experiment.json", {"team": TEAM_NAME, "members": TEAM_MEMBERS,
    "mode": MODE, "run_name": RUN_NAME, "trust": trust_metrics,
    "training_index": training_index_stats, "pairs": pair_stats})
register_run(RUN_DIR, WORK_ROOT / "registry")
trust_metrics


In [ ]:
if EVALUATE_TRUST:
    with (RUN_DIR / "trust_predictions.jsonl").open(encoding="utf-8") as f:
        worst = sorted((json.loads(line) for line in f), key=lambda r: r["f05"])[:10]
    print([{"id": r["id"], "country": r["country"], "score": r["f05"], "errors": r["errors"]} for r in worst])
else:
    print("Trust cases remain unopened. Select settings using calibration reports.")


## 8. Geographic stress tests (enabled by default)

These fit new models excluding one country's training labels, probability-calibration labels and threshold-selection labels, then evaluate that country's trust anchors. The lexical index still contains all unlabeled target text. These tests reveal geographic fragility; they do not provide a French score.

In [ ]:
stress_metrics = {}
if RUN_GEOGRAPHIC_STRESS:
    for held_country in ["US", "India"]:
        stress_root = RUN_DIR / ("without_" + held_country)
        stress_root.mkdir(exist_ok=True)
        stress_language = None
        if USE_MINED_LANGUAGE:
            stress_language = stress_root / "language.json"
            if not stress_language.exists():
                mine_language(DATA_ROOT / "train/train_source1.tsv", DATA_ROOT / "train/train_ground_truth.tsv",
                    [DATA_ROOT / "train" / f"train_source{s}.tsv" for s in (2, 3)], stress_language,
                    per_country=PER_COUNTRY, exclude_country=held_country)
        build_index([DATA_ROOT / "train" / f"train_source{s}.tsv" for s in (2, 3)],
                    stress_root / "index", stress_language)
        prepare_pairs(DATA_ROOT / "train/train_source1.tsv", DATA_ROOT / "train/train_ground_truth.tsv",
                      stress_root / "index", stress_root, retrieval=retrieval,
                      per_country=PER_COUNTRY, max_pairs=MAX_TRAINING_PAIRS)
        stress_model = stress_root / f"model_without_{held_country}.pkl"
        if not stress_model.exists():
            train(stress_root, max_iter=MAX_ITER, exclude_country=held_country)
        if EVALUATE_TRUST:
            stress_metrics[held_country] = evaluate_trust(stress_root, stress_model, country=held_country)
        else:
            stress_metrics[held_country] = "model ready; trust unopened"
stress_metrics


## 9. Build the complete test index and measure throughput

The test index includes France automatically. The benchmark writes into a separate directory and cannot be packaged as a final submission. For a meaningful full-scale estimate, use several thousand anchors spanning the countries and inspect cold/warm runtime and long candidate tails. The benchmark samples country reservoirs across the entire input, then mixes them in proportion to country populations while ensuring country coverage when the limit permits.

The estimate below concerns inference only; add index creation, training, validation, export and upload time. Full AWS throughput and storage requirements must be measured on your actual instance.

In [ ]:
estimated_hours = None
if not TRAIN_ONLY:
    test_index_stats = build_index([DATA_ROOT / "test" / f"test_source{s}.tsv" for s in (2, 3)], TEST_INDEX, language_file)
    if RUN_FALLBACK:
        fallback(DATA_ROOT / "test/test_source1.tsv", TEST_INDEX, RUN_DIR / "fallback", workers=WORKERS)
        validate(DATA_ROOT / "test/test_source1.tsv", TEST_INDEX, RUN_DIR / "fallback")
    benchmark_dir = RUN_DIR / ("benchmark_workers_" + str(WORKERS))
    if (benchmark_dir / "inference_manifest.json").exists():
        benchmark = json.loads((benchmark_dir / "inference_manifest.json").read_text())
    else:
        benchmark = predict(DATA_ROOT / "test/test_source1.tsv", TEST_INDEX, MODEL_PATH,
                            benchmark_dir, batch_size=100, limit=BENCHMARK_ANCHORS, workers=WORKERS)
    assert benchmark.get("complete") and benchmark.get("valid_fresh_benchmark"), "Use a new benchmark directory for fresh timing"
    assert benchmark["limit"] == BENCHMARK_ANCHORS and benchmark["model_sha256"] == file_hash(MODEL_PATH), "Benchmark settings changed; use a new directory"
    assert benchmark["source1_sha256"] == file_hash(DATA_ROOT / "test/test_source1.tsv") and benchmark["index_fingerprint"] == test_index_stats["fingerprint"], "Benchmark data changed"
    TEST_ANCHORS = sum(1 for _ in records(DATA_ROOT / "test/test_source1.tsv"))
    rate = benchmark["anchors"] / max(benchmark["seconds_this_call"], .001)
    estimated_hours = 1.35 * TEST_ANCHORS / rate / 3600
    print({"anchors_per_second": rate, "estimated_hours": estimated_hours, "drift": benchmark["drift"]})
else:
    print("Train-only mode: no official test files opened.")


## 10. Full inference and strict validation

Enable `RUN_FULL_TEST` in configuration after reviewing the estimate. Each completed batch contains both files and checksums; rerunning resumes compatible batches. Changed models, input files, retrieval settings or batch sizes require a new output directory.

The strict validator checks every test anchor exactly once, target existence, duplicates, country agreement, and prediction ⊆ candidate. The candidate file contains **exactly the records scored by the final matcher**, after retrieval pruning. Both files include empty rows where appropriate.

In [ ]:
validation_result = None
if RUN_FULL_TEST and not TRAIN_ONLY:
    if estimated_hours > INFERENCE_BUDGET_HOURS and not ALLOW_RUNTIME_OVERRUN:
        raise RuntimeError("Inference estimate exceeds budget; benchmark before proceeding.")
    prediction_stats = predict(DATA_ROOT / "test/test_source1.tsv", TEST_INDEX, MODEL_PATH,
                               OUTPUT_DIR, batch_size=500, workers=WORKERS)
    validation_result = validate(DATA_ROOT / "test/test_source1.tsv", TEST_INDEX, OUTPUT_DIR)
    if RUN_OFFICIAL_VALIDATOR:
        official_validate(OFFICIAL_VALIDATOR, DATA_ROOT / "test", OUTPUT_DIR)
validation_result or "Full inference disabled."


## 11. Fill your methodology, package, and optionally upload to your S3 bucket

Edit `Documentation_template.md` with team details and **real-data** results. Packaging refuses unfilled markers. Demo output should never be uploaded to the competition. The ZIP includes the final outputs, self-contained source, model and inference configuration. This notebook does not upload to the competition portal.

S3 upload is an explicit optional transport step using the notebook's existing IAM role. Shut down your AWS compute when finished; this notebook does not manage instance lifecycles.

In [ ]:
FINAL_ZIP = RUN_DIR / "team_submission.zip"
if PACKAGE_SUBMISSION:
    assert MODE == "full" and not TRAIN_ONLY
    assert validation_result and validation_result["status"] == "PASS"
    assert (OUTPUT_DIR / "official_validation.json").exists(), "Run the supplied official validator with --check-ids"
    license_inventory(RUN_DIR / "dependency_licenses.json")
    package_submission(PROJECT_ROOT, OUTPUT_DIR, MODEL_PATH, FINAL_ZIP, require_official=True)
    print(FINAL_ZIP)
if UPLOAD_PACKAGE_TO_S3:
    assert PACKAGE_SUBMISSION and S3_OUTPUT_URI
    from ber.aws import upload
    upload(FINAL_ZIP, S3_OUTPUT_URI)


## What to improve next

1. Compare retrieval-channel contributions and candidate budgets on the same anchors. Missing true links cannot be repaired by a classifier.
2. Add training-fold-only alias/transliteration learning or a separately licensed multilingual retriever for cross-script failures. Current script preservation plus address retrieval is **not transliteration**.
3. Test a stronger anchor-level utility decoder and near-duplicate cluster splits once the measured baseline is stable.

Implementation references: [SQLite FTS5](https://www.sqlite.org/fts5.html), [scikit-learn histogram gradient boosting](https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html), [Boto3 role-based credentials](https://docs.aws.amazon.com/boto3/latest/guide/credentials.html). These references concern software behavior; no external business-identity data is used.